# Решения: Проект 16 — A/B-тестирование

Решения упражнений из `16_AB_Testing.ipynb`. Ноутбук самодостаточен: данные генерируются тем же способом, что и в проекте.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# --- воспроизводим данные проекта 16 ---
np.random.seed(42)
true_conv_A, true_conv_B = 0.120, 0.138
n_A = n_B = 5000
conv_A = np.random.binomial(1, true_conv_A, n_A)
conv_B = np.random.binomial(1, true_conv_B, n_B)
time_A = np.random.lognormal(4.8, 0.5, n_A)
time_B = np.random.lognormal(4.9, 0.5, n_B)
df = pd.DataFrame({
    'user_id': np.arange(1, n_A + n_B + 1),
    'group': ['A'] * n_A + ['B'] * n_B,
    'converted': np.concatenate([conv_A, conv_B]),
    'time_on_site': np.round(np.concatenate([time_A, time_B]), 1),
}).sample(frac=1, random_state=42).reset_index(drop=True)

s_A = df[df.group == 'A']['converted'].sum()
s_B = df[df.group == 'B']['converted'].sum()
p_A, p_B = s_A / n_A, s_B / n_B
alpha = 0.05
print(f'p_A = {p_A:.4f}, p_B = {p_B:.4f}, разница = {(p_B - p_A)*100:.2f} п.п.')

## Упражнение 1: Односторонний тест

$H_0: p_B = p_A$ против $H_1: p_B > p_A$. Для одностороннего теста p-значение равно **половине** двустороннего (когда наблюдаемая разница в сторону $H_1$).

In [ ]:
p_pool = (s_A + s_B) / (n_A + n_B)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
z = (p_B - p_A) / se

p_two = 2 * (1 - stats.norm.cdf(abs(z)))
p_one = 1 - stats.norm.cdf(z)   # H1: p_B > p_A

print(f'z-статистика: {z:.4f}')
print(f'p-значение (двустороннее): {p_two:.5f}')
print(f'p-значение (одностороннее): {p_one:.5f}')
print(f'\nОдностороннее = двустороннее / 2: {np.isclose(p_one, p_two/2)}')
print('Вывод:', 'отвергаем H0' if p_one < alpha else 'не отвергаем H0',
      '- эффект B статистически значим' if p_one < alpha else '')
print('Односторонний тест мощнее, но применим только при обоснованном')
print('априорном ожидании направления эффекта (B не хуже A).')

## Упражнение 2: Анализ вторичной метрики (время на сайте)

Сравним среднее время на сайте t-тестом Уэлча (не предполагаем равные дисперсии) и построим доверительный интервал для разницы средних.

In [ ]:
ta = df[df.group == 'A']['time_on_site']
tb = df[df.group == 'B']['time_on_site']

t_stat, p_val = stats.ttest_ind(tb, ta, equal_var=False)

# 95% ДИ для разницы средних (Уэлч)
diff = tb.mean() - ta.mean()
se_diff = np.sqrt(ta.var(ddof=1)/len(ta) + tb.var(ddof=1)/len(tb))
dfree = (se_diff**4) / ((ta.var(ddof=1)/len(ta))**2/(len(ta)-1) +
                        (tb.var(ddof=1)/len(tb))**2/(len(tb)-1))
tcrit = stats.t.ppf(0.975, dfree)
ci = (diff - tcrit*se_diff, diff + tcrit*se_diff)

print(f'Среднее время A: {ta.mean():.1f} c | B: {tb.mean():.1f} c')
print(f't-статистика: {t_stat:.3f}, p-значение: {p_val:.5f}')
print(f'Разница средних: {diff:.2f} c')
print(f'95% ДИ для разницы: [{ci[0]:.2f}, {ci[1]:.2f}] c')
print('Значимо:' , 'да' if p_val < alpha else 'нет',
      '(ДИ не включает 0)' if ci[0] > 0 or ci[1] < 0 else '(ДИ включает 0)')

## Упражнение 3: Планирование размера выборки

Формула размера выборки на группу для сравнения двух пропорций:

$$n = \frac{(z_{1-\alpha/2} + z_{1-\beta})^2\,[p_1(1-p_1) + p_2(1-p_2)]}{(p_2 - p_1)^2}$$

Целевой эффект **+1.5 п.п.** от базовой 12% при мощности 80%.

In [ ]:
def sample_size(p1, p2, power=0.8, alpha=0.05):
    z_a = stats.norm.ppf(1 - alpha/2)
    z_b = stats.norm.ppf(power)
    num = (z_a + z_b)**2 * (p1*(1-p1) + p2*(1-p2))
    return int(np.ceil(num / (p2 - p1)**2))

p1, p2 = 0.12, 0.135  # +1.5 п.п.
n_need = sample_size(p1, p2, power=0.8)
print(f'Нужно ~{n_need:,} пользователей на группу (эффект +1.5 п.п., мощность 80%).')

# Проверка через statsmodels
try:
    from statsmodels.stats.power import NormalIndPower
    from statsmodels.stats.proportion import proportion_effectsize
    es = proportion_effectsize(p2, p1)
    n_sm = NormalIndPower().solve_power(es, power=0.8, alpha=0.05, alternative='two-sided')
    print(f'statsmodels: ~{np.ceil(n_sm):,.0f} на группу (для сверки).')
except Exception as e:
    print('statsmodels недоступен:', e)

## Упражнение 4: Влияние априорного распределения

Повторим байесовский Beta-Binomial с информативным априорным $\text{Beta}(12, 88)$ (ожидание конверсии ~12%) и посмотрим на устойчивость при малой выборке (первые 200).

In [ ]:
def bayes_ab(sA, nA, sB, nB, a0, b0, seed=7, n=100000):
    postA = stats.beta(a0 + sA, b0 + nA - sA)
    postB = stats.beta(a0 + sB, b0 + nB - sB)
    rng = np.random.default_rng(seed)
    xa, xb = postA.rvs(n, random_state=rng), postB.rvs(n, random_state=rng)
    return (xb > xa).mean()

# Полные данные: неинформативный vs информативный априор
p_flat = bayes_ab(s_A, n_A, s_B, n_B, 1, 1)
p_info = bayes_ab(s_A, n_A, s_B, n_B, 12, 88)
print('Полные данные (5000/группа):')
print(f'  Beta(1,1):   P(B>A) = {p_flat*100:.2f}%')
print(f'  Beta(12,88): P(B>A) = {p_info*100:.2f}%  (априор почти не влияет)')

# Малая выборка: первые 200 пользователей
small = df.head(200)
sa = small[small.group=='A']['converted'].sum(); na = (small.group=='A').sum()
sb = small[small.group=='B']['converted'].sum(); nb = (small.group=='B').sum()
ps_flat = bayes_ab(sa, na, sb, nb, 1, 1)
ps_info = bayes_ab(sa, na, sb, nb, 12, 88)
print(f'\nМалая выборка (первые 200: A={na}, B={nb}):')
print(f'  Beta(1,1):   P(B>A) = {ps_flat*100:.2f}%')
print(f'  Beta(12,88): P(B>A) = {ps_info*100:.2f}%  (априор влияет заметно сильнее)')
print('\nВывод: при малых данных выбор априорного распределения критичен;')
print('при больших данных правдоподобие доминирует и априор почти не важен.')